<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: classification.**

Building on the "Answered Away" project from Week 1, this is a page-level classification
problem: given a page's prior-period signals, predict which of three classes it falls into —
`answered_away`, `normal_decay`, or `stable_other`. It's not ranking, since I'm not ordering
pages against each other, and it's not clustering, since I already have a rule-based way to
assign the label after the fact from Week 1 — the question here is whether that label can be
*predicted* from signals available before the drop fully shows up, not *discovered* from
scratch.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target**: the `pattern_group` column from Week 1 (`answered_away` / `normal_decay` /
`stable_other`), built from `impr_change_pct` and `click_change_pct` on the last-30-vs-prev-30
day windows.

**Where the label comes from**: a defined rule, not an observed outcome. No column in this
dataset confirms what actually appeared on the search results page for any given query, so
this is a proxy label — a behavioral pattern consistent with click suppression, not a
confirmed cause. My "ground truth" is itself a rule-based proxy, so the model is predicting a
proxy of a proxy, and I want to be upfront about that rather than treat the label as more
solid than it is.

## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Likely metric**: precision/recall or F1 on the `answered_away` class specifically, not plain
accuracy, since the classes are probably imbalanced and accuracy would be misleading if
`answered_away` is the minority class. Recall matters because missing a real answered_away
page means it keeps losing clicks under the wrong fix. Precision matters because a false
positive wastes an editor's time on the wrong intervention. I'll confirm the actual class
counts in section 4 and lock in the metric based on the real numbers rather than an assumption.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis**: one row = one page (`content_id`), same as Week 1.

I'm loading the same filtered subset from Week 1 (pages with enough prior volume to trust the
percent change) and showing the real `pattern_group` counts, since the class balance here is
what determines the metric I lock in above.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
import pandas as pd

# Load the starter CSV from the raw GitHub URL (relative paths don't work in Colab —
# the repo isn't cloned into the session, same fix as Week 1)
df = pd.read_csv('https://raw.githubusercontent.com/mehanshbarthwal-lab/search-ranking-ml/main/data/raw/content_refresh_anonymized.csv')

# Filter to pages with enough prior volume BEFORE computing pct change, to avoid divide-by-zero
volume_mask = (df['impressions_prev_30d'] >= 50) & (df['clicks_prev_30d'] >= 3)
df_filtered = df[volume_mask].copy()

# Compute percentage changes only on the filtered, safe subset
df_filtered['impr_change_pct'] = (
    (df_filtered['impressions_last_30d'] - df_filtered['impressions_prev_30d'])
    / df_filtered['impressions_prev_30d'] * 100
)
df_filtered['click_change_pct'] = (
    (df_filtered['clicks_last_30d'] - df_filtered['clicks_prev_30d'])
    / df_filtered['clicks_prev_30d'] * 100
)

# Define the pattern groups (same rule as Week 1)
def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df_filtered['pattern_group'] = df_filtered.apply(assign_pattern, axis=1)

# Unit of analysis check: one row = one page (content_id)
print("Unit of analysis — one row per page:")
print(df_filtered[['content_id', 'pattern_group']].head())
print()

# Class balance — this is what decides the real success metric in section 3
print("Total pages in each pattern group:")
print(df_filtered['pattern_group'].value_counts())
print()
print("Class balance (%):")
print((df_filtered['pattern_group'].value_counts(normalize=True) * 100).round(2))

Unit of analysis — one row per page:
              content_id pattern_group
0   content_304f48230142  normal_decay
2   content_9aa793d4d895  normal_decay
3   content_331d6c4de07b  stable_other
8   content_5e6c160719bc  stable_other
10  content_d8ee6cc6d642  stable_other

Total pages in each pattern group:
pattern_group
stable_other     3360
normal_decay     2693
answered_away     641
Name: count, dtype: int64

Class balance (%):
pattern_group
stable_other     50.19
normal_decay     40.23
answered_away     9.58
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.